# 02 - Integration & Clustering

**Goal:** combine the four samples, correct for batch effects, and group similar cells into clusters that we'll name in notebook 03.

**The batch problem:** if you just merge the samples, cells often cluster by *which tube they came from* instead of by *what kind of cell they are*. Batch correction (Harmony here) removes that technical signal while keeping the biology. We compare before/after so you can show it worked.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))   # make the src/ package importable
from src import config as cfg
from src import utils as ut
ut.setup_scanpy()

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
print("scanpy", sc.__version__)

In [ ]:
adata = ut.load_checkpoint('01_qc_done')
adata

## PCA, then a neighbour graph + UMAP WITHOUT correction (the 'before' picture)

In [ ]:
sc.pp.pca(adata, n_comps=cfg.N_PCS, random_state=cfg.RANDOM_SEED)
sc.pp.neighbors(adata, n_neighbors=cfg.N_NEIGHBORS, n_pcs=cfg.N_PCS)
sc.tl.umap(adata, random_state=cfg.RANDOM_SEED)
sc.pl.umap(adata, color='sample', title='Before integration',
           save='_02_before_integration.png')

## Run Harmony, rebuild the graph on the corrected space (the 'after' picture)
If samples were previously separated by batch, they should now overlap where the same cell types are present.

In [ ]:
sc.external.pp.harmony_integrate(adata, key=cfg.BATCH_KEY)
sc.pp.neighbors(adata, n_neighbors=cfg.N_NEIGHBORS, n_pcs=cfg.N_PCS,
                use_rep='X_pca_harmony')
sc.tl.umap(adata, random_state=cfg.RANDOM_SEED)
sc.pl.umap(adata, color='sample', title='After integration (Harmony)',
           save='_02_after_integration.png')

## Leiden clustering
`resolution` in config controls granularity: higher = more, smaller clusters. Tune it so the cluster count roughly matches the number of cell types you expect in PBMCs (~8-12).

In [ ]:
sc.tl.leiden(adata, resolution=cfg.LEIDEN_RES, random_state=cfg.RANDOM_SEED)
sc.pl.umap(adata, color='leiden', legend_loc='on data',
           save='_02_leiden_clusters.png')
adata.obs['leiden'].value_counts()

In [ ]:
ut.save_checkpoint(adata, '02_clustered')